# Simple LLM Pipeline (Minimal GPT from Scratch)
This notebook shows the **simplest end‑to‑end pipeline** for a tiny LLM:
1. Load text
2. Tokenize
3. Build embeddings
4. Implement attention
5. Build a tiny GPT model
6. Train on next‑token prediction
7. Generate text

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


## 1. Load Simple Training Text

In [2]:
text = """
Machine learning is fun.
Deep learning builds powerful models.
Language models learn patterns in text.
"""


In [3]:
# Use real text for better Vocabulary
import requests

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text

## 2. Tokenization (character level for simplicity)

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(vocab_size)


65


## 3. Create Training Batches

In [5]:
# block_size = 8
block_size = 16
# Increase Context Window

def get_batch():
    ix = torch.randint(len(data)-block_size, (4,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y


## 4. Self‑Attention

In [6]:
class SelfAttention(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.key = nn.Linear(n_embd, n_embd)
        self.query = nn.Linear(n_embd, n_embd)
        self.value = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        K = self.key(x)
        Q = self.query(x)
        V = self.value(x)

        scores = Q @ K.transpose(-2,-1) / math.sqrt(x.size(-1))
        weights = F.softmax(scores, dim=-1)

        return weights @ V


## 5. Transformer Block

In [7]:
class Block(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.attn = SelfAttention(n_embd)
        self.ff = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd)
        )

    def forward(self, x):
        x = x + self.attn(x)
        x = x + self.ff(x)
        return x


## 6. Tiny GPT Model

In [8]:
class TinyGPT(nn.Module):
    # def __init__(self, vocab_size, n_embd=32):
    def __init__(self, vocab_size, n_embd=64):
        # Slightly Bigger Model
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)

        self.block = Block(n_embd)
        self.ln = nn.LayerNorm(n_embd)

        self.head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):

        B,T = idx.shape

        tok = self.token_emb(idx)
        pos = self.pos_emb(torch.arange(T))

        x = tok + pos

        x = self.block(x)

        x = self.ln(x)

        logits = self.head(x)

        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss


## 7. Train Model

In [9]:
model = TinyGPT(vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# from range 500 to 5000, mean train longer
for step in range(5000):

    xb, yb = get_batch()

    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(step, loss.item())


0 4.3342814445495605
100 2.686142683029175
200 0.9585402011871338
300 0.3873348832130432
400 0.2559193968772888
500 0.16928941011428833
600 0.21416860818862915
700 0.3041740357875824
800 0.16812799870967865
900 0.22896166145801544
1000 0.14150512218475342
1100 0.2040439248085022
1200 0.25577598810195923
1300 0.12936489284038544
1400 0.19501258432865143
1500 0.18223856389522552
1600 0.1648171842098236
1700 0.17128339409828186
1800 0.18705961108207703
1900 0.287144273519516
2000 0.18696124851703644
2100 0.18372982740402222
2200 0.22518236935138702
2300 0.1592252105474472
2400 0.17805668711662292
2500 0.1275235116481781
2600 0.09989452362060547
2700 0.20786666870117188
2800 0.13414448499679565
2900 0.12570953369140625
3000 0.12846170365810394
3100 0.1920367181301117
3200 0.16542229056358337
3300 0.1215573102235794
3400 0.1571221500635147
3500 0.22855505347251892
3600 0.1273432821035385
3700 0.1257447600364685
3800 0.19560371339321136
3900 0.2058231383562088
4000 0.16547638177871704
4100 0

## 8. Generate Text

In [10]:
def generate(model, start, length=100):

    idx = torch.tensor([encode(start)], dtype=torch.long)

    for _ in range(length):

        idx_cond = idx[:,-block_size:]

        logits,_ = model(idx_cond)

        logits = logits[:,-1,:]

        probs = F.softmax(logits, dim=-1)

        next_id = torch.multinomial(probs,1)

        idx = torch.cat((idx,next_id),dim=1)

    return decode(idx[0].tolist())


print(generate(model, "Machine"))


Machine;feieimm!r woneais p-gge wave, : avEll.

IOMIOUO:
MNit es fowis y. d
F ave shy har, iromy nf r ag re


Notice what the model already learned:

Capital letters starting lines

Apostrophes (I'ran)

Sentence punctuation

Word-like shapes

Line breaks like dialogue